In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os
from shapely.geometry import Point
import shapely
gpd.__version__

In [ ]:
outfile = "coast_v0-1-0"

faults_file = f"./QGIS/CFM_active_faults.shp"
faults = gpd.read_file(faults_file)

islands_file = f"./data./coastline./nz-coastlines-and-islands-polygons-topo-150k.gpkg"

low_res_coast = False
if low_res_coast:
    islands_file = islands_file.replace("150k", "1500k")
    coast_res_tag = "_lowRes"
else:
    coast_res_tag = ""


os.makedirs("./sites/poly_buffers", exist_ok=True)
os.makedirs("./sites/N-S_sites", exist_ok=True)
os.makedirs("./sites/coastline_sites", exist_ok=True)
os.makedirs("./sites/coastline_fault_sites", exist_ok=True)

islands = gpd.read_file(islands_file)
min_island = 10 # km2

coast_resolution = 2000

# Buffers around Fault Trace only
fault_buffers = [[500, 2500, 0.35], [1000, 5000, 0.00]]  # [Resolution (m), Buffer (m), minSlipRate (mm/yr)]
# fault_buffers = []  # [Resolution (m), Buffer (m), minSlipRate (mm/yr)]

# Buffers around interface polygons
interface_polygons = [["sz", 5000, 0, 2000, 1000], ["py", 5000, 0, 2000, 1000]]  # [Interface, hanging buff (m), foot buff (m), edge buff (m), resolution]
# interface_polygons = []  # [Interface, hanging buff (m), foot buff (m), edge buff (m), resolution]

coast_crop_file = None
# coast_crop_file = f"./sites./poly_buffers./coastline_8000m_dilation_1500m_buffer_gt170sqkmIslands.geojson"

xmin, ymin, xmax, ymax = faults.total_bounds
if low_res_coast:
    islands = islands[islands["area"] >= 1e6 * min_island]
else:
    islands = islands.cx[xmin:xmax, ymin:ymax]
    islands = islands[islands["Area_SquareMeter"] >= 1e6 * min_island]
islands.drop(islands[islands.name == "Antipodes Island"].index)
coastline = islands.geometry.boundary.unary_union

In [ ]:
# Default Coastline
print(f"{coast_resolution} m resolution around {islands.shape[0]} islands >= {min_island} km2...", end="\r")
n_sites = []
site_file = f"./sites/coastline_sites/coastline{coast_res_tag}_{coast_resolution}m_{min_island}km.geojson"
if os.path.exists(site_file):
    coast_site_gdfs = [gpd.read_file(site_file)]
else:
    distances = np.arange(0, coastline.length, coast_resolution)
    coast_site_gdfs = [gpd.GeoDataFrame({'resolution': coast_resolution, 'geometry': coastline.interpolate(distances)}, geometry='geometry', crs=2193)]
    coast_site_gdfs[0].to_file(site_file, driver="GeoJSON")
n_sites.append(coast_site_gdfs[0].shape[0])
print(f"{coast_resolution} m resolution around {islands.shape[0]} islands >= {min_island} km2: {n_sites[-1]} sites")

In [ ]:
# Fault Buffers
for resolution, buffer, sliprate in fault_buffers:
    site_file = f"./sites/coastline_fault_sites/coast{coast_res_tag}_CFM_{sliprate}mmyr_b{buffer}m_r{resolution}m_{min_island}km2.geojson"
    if os.path.exists(site_file):
        print("Loaded", site_file)
        coast_site_gdfs.append(gpd.read_file(site_file))
    else:
        print(f"{resolution} m resolution...", end="\r")
        coast_file = f"./sites/coastline_sites/coastline{coast_res_tag}_{resolution}m_{min_island}km2.geojson"
        if os.path.exists(coast_file):
            coast_points = gpd.read_file(coast_file)
        else:
            distances = np.arange(0, coastline.length, resolution)
            coast_points = gpd.GeoSeries(coastline.interpolate(distances), crs=2193)
            gpd.GeoDataFrame({'resolution': coast_resolution, 'geometry': coastline.interpolate(distances)}, geometry='geometry', crs=2193).to_file(coast_file, driver="GeoJSON")
        print(f"{resolution} m resolution around >{sliprate} mm/yr faults...", end="\r")
        trimmed_faults = faults[faults.SR_pref > sliprate]
        print(f"{resolution} m resolution around >{sliprate} mm/yr faults for {buffer} m...", end="\r")
        buffer_file = f"./sites/poly_buffers/CFM_trace_{sliprate}mmyr_b{buffer}m.geojson"
        if os.path.exists(buffer_file):
            buffers = gpd.read_file(buffer_file)['geometry']
        else:
            buffers = faults[faults.SR_pref > sliprate].buffer(buffer)
            gpd.GeoDataFrame(geometry=buffers, crs=2193).to_file(buffer_file, driver="GeoJSON")
        coast_points = coast_points[coast_points.within(buffers.unary_union)]
        coast_site_gdfs.append(pd.DataFrame({'resolution': [resolution] * coast_points.shape[0], 'geometry': coast_points.geometry}))
        gpd.GeoDataFrame(coast_site_gdfs[-1], geometry='geometry', crs=2193).to_file(site_file, driver="GeoJSON")
    n_sites.append(coast_site_gdfs[-1].shape[0])
    print(f"{resolution} m resolution around >{sliprate} mm/yr faults for {buffer} m: {n_sites[-1]} sites")

In [ ]:
# Interface Buffers
for fault_type, hangingbuff, footbuff, edgebuff, resolution in interface_polygons:
    site_file = f"./sites/coastline_fault_sites/{fault_type}_hang-{hangingbuff}m_foot-{footbuff}m_edge-{edgebuff}m_r{resolution}m.geojson"
    if os.path.exists(site_file):
        print("Loaded", site_file)
        coast_site_gdfs.append(gpd.read_file(site_file))
    else:
        polyname = f"./sites/poly_buffers/{fault_type}-plane_hang-{hangingbuff}m_foot-{footbuff}m_edge-{edgebuff}m.geojson"
        if not os.path.exists(polyname):
            print(f"[ERROR]: {polyname} not found")
            continue
        coast_file = f"./sites/coastline_sites/coastline_{resolution}m.geojson"
        print(f"{resolution} m resolution...", end="\r")
        if os.path.exists(coast_file):
            coast_points = gpd.read_file(coast_file).geometry
        else:
            distances = np.arange(0, coastline.length, resolution)
            coast_points = gpd.GeoSeries(coastline.interpolate(distances), crs=2193)
            gpd.GeoDataFrame({'resolution': coast_resolution, 'geometry': coast_points}, geometry='geometry', crs=2193).to_file(coast_file, driver="GeoJSON")
        print(f"{resolution} m resolution around {fault_type}_hang-{hangingbuff}m_foot-{footbuff}m_edge-{edgebuff}m...", end="\r")
        polys = gpd.read_file(polyname).unary_union
        coast_points = coast_points[gpd.GeoSeries(coast_points, crs=2193).within(polys)]
        coast_site_gdfs.append(gpd.GeoDataFrame({'resolution': resolution, 'geometry': coast_points.geometry}, geometry='geometry', crs=2193))
        coast_site_gdfs[-1].to_file(site_file, driver="GeoJSON")
    n_sites.append(coast_site_gdfs[-1].shape[0])
    print(f"{resolution} m resolution around {fault_type}_hang-{hangingbuff}m_foot-{footbuff}m_edge-{edgebuff}m: {n_sites[-1]} sites")


In [ ]:
coast_site_gdf = gpd.GeoDataFrame(pd.concat(coast_site_gdfs, ignore_index=True), geometry='geometry', crs=2193)
n_sites.append(coast_site_gdf.shape[0])
print(f"Dropping Duplicates sites: {n_sites[-1]} -> ...", end="\r")
coast_site_gdf = coast_site_gdf.sort_values('resolution').drop_duplicates(ignore_index=True, keep='first', subset="geometry").reset_index(drop=True)
n_sites.append(coast_site_gdf.shape[0])
print(f"Dropping Duplicates sites: {n_sites[-2]} -> {n_sites[-1]}")
coast_site_gdf.geometry = shapely.set_precision(coast_site_gdf.geometry.array, grid_size=1)

In [ ]:
coast_site_gdf

In [ ]:
coast_crop = False
if coast_crop_file:
    if os.path.exists(coast_crop_file):
        print(f"Cropping sites to {os.path.basename(coast_crop_file)}: {n_sites[-1]} -> ...", end="\r")
        coast_buffer = gpd.read_file(coast_crop_file)
        coast_site_gdf = coast_site_gdf[coast_site_gdf.intersects(coast_buffer.unary_union)].reset_index(drop=True)
        n_sites.append(coast_site_gdf.shape[0])
        print(f"Cropping sites to {os.path.basename(coast_crop_file)}: {n_sites[-2]} -> {n_sites[-1]}")
        coast_crop = True
    else:
        print(f"[ERROR]: {coast_crop_file} not found")  

In [ ]:
print(f"Writing to ./sites/{outfile}.geojson")
coast_site_gdf.geometry = shapely.set_precision(coast_site_gdf.geometry, grid_size=1)
coast_site_gdf['siteId'] = coast_site_gdf.geometry.apply(lambda x: f"{int(x.x)}_{int(x.y)}")
print('Formatting....')
coast_site_gdf = coast_site_gdf.reset_index(drop=True)
coast_site_gdf['Lon'] = coast_site_gdf.geometry.x
coast_site_gdf['Lat'] = coast_site_gdf.geometry.y
coast_site_gdf['Height'] = 0

coast_site_gdf.to_file(f"./sites/{outfile}.geojson", drive="GeoJSON")

print("Splitting outputs into Hikurangi and Puysegur sections...")
wellington = Point([1749150, 5428092]) # Wellington coordinates in NZTM
te_anau = Point([1186710, 4957633])  # Te Anau coordinates in NZTM
distance = 350  # Distance South of Wellington in km to include for hikurangi

# For Hikurangi, find all centroids north of 350km south of Wellington
northern_section = coast_site_gdf[(coast_site_gdf.geometry.y > wellington.y) | (coast_site_gdf.distance(wellington) < distance * 1e3)]
if northern_section.shape[0] > 0:
    northern_section.to_file(f"./sites/N-S_sites/{outfile}N.geojson", driver='GeoJSON')
    print(f"\t{northern_section.shape[0]} Northern sites in {outfile}N.geojson")

# For Puysegur, find all centroids within 350km of Te Anau
southern_section = coast_site_gdf[(coast_site_gdf.distance(te_anau) < distance * 1e3)]
if southern_section.shape[0] > 0:
    southern_section.to_file(f"./sites/N-S_sites/{outfile}S.geojson", driver='GeoJSON')
    print(f"\t{southern_section.shape[0]} Southern sites in {outfile}S.geojson")

In [ ]:
print(f"Writing metadata for {outfile}")
with open(f"./sites/{outfile}_meta.txt", 'w') as f:
    f.write(f"Name: {outfile}\n\n")
    f.write(f"Coast File: {islands_file} ({islands.shape[0]} islands >= {min_island} km2)\n")
    f.write(f"National Resolution: {coast_resolution} km ({n_sites[0]})\n")
    if len(fault_buffers) > 1 and len(fault_buffers[0]) == 3:
        f.write(f"\nFault Buffers:\n")
        for fix, (resolution, buffer, sliprate) in enumerate(fault_buffers, 1):
            f.write(f"\t{fix}: {resolution} m resolution around >{sliprate} mm/yr faults for {buffer} m ({n_sites[fix]})\n")
    else:
        fix = 0

    if len(interface_polygons) > 1 and len(interface_polygons[0]) == 5:
        f.write(f"\nInterface Buffers:\n")
        for ix, (fault_type, hangingbuff, footbuff, edgebuff, resolution) in enumerate(interface_polygons, fix + 1):
            f.write(f"\t{ix}: {resolution} m resolution around {fault_type}_hang-{hangingbuff}km_foot-{footbuff}km_edge-{edgebuff}km: {n_sites[ix]} sites\n")

    if coast_crop_file:
        if coast_crop:  # Successful
            f.write(f"\nDuplicate sites: {n_sites[-3] - n_sites[-2]} removed\n")
            f.write(f"\nCoast Buffer Crop: {coast_crop_file} ({n_sites[-2] - n_sites[-1]} removed)\n")
        else: # Unsuccessful
            f.write(f"\nDuplicate sites: {n_sites[-2] - n_sites[-1]} removed\n")
            f.write(f"\nCoast Buffer Crop: {coast_crop_file} [FILE NOT FOUND]\n")

    else: 
        f.write(f"\nDuplicate sites: {n_sites[-2] - n_sites[-1]} removed\n")
    f.write(f"Total sites: {n_sites[-1]}\n")
    f.write(f"Northern sites: {northern_section.shape[0]}\n")
    f.write(f"Southern sites: {southern_section.shape[0]}\n")